In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import json

In [2]:
OUTPUT_PATH = Path("dati_schede_min.csv")
df_new = pd.read_csv(OUTPUT_PATH, dtype=str, keep_default_na=False)
df_new.head()

,id_scheda_originale,numero_catalogo_iccd,categoria_edificio,tipologia_edificio,numero_spazi,regione,sigla_provincia,comune,contesto_paesaggistico,tipo_dissesto_idrogeologico,...,anno_compilazione_scheda,nome_compilatore,ente_compilatore,nome_responsabile_scientifico,ruolo_responsabile_scientifico,ente_responsabile_scientifico,profilo_pubblicazione,titolo,descrizione,gaussian
0,ce9c9265-54dd-4779-9c4b-cc282dc908e3,182407,EDIFICIO SINGOLO,casa con loggiato,nr,Friuli-Venezia Giulia,UD,Dignano,pianura,alluvione,...,2024.0,"Rulli, Eduardo",Atlante Group SPA,"Parisi, Valeria",responsabile coordinamento delle attività,Atlante Group s.r.l.,2.0,,"A un’attenta osservazione del prospetto sud, l...",0
1,6e02ff7c-2b9d-40d0-a14c-e8f4d03d9a83,252733,EDIFICIO SINGOLO,casa a corte chiusa,nr,Sardegna,SU,Soleminis,collina,nessun dissesto evidente,...,2025.0,"Camatti, Matteo",Progetto PSC,"cesarano, barbara",responsabile coordinamento delle attività,progetto psc,1.0,,Casa a corte retrostante ad uso abitativo (1a-...,0
2,814d64dd-d0f5-4c01-ace8-968ec355547e,223849,EDIFICIO CON ANNESSI,casa a scala esterna,1,Basilicata,PZ,San Martino d'Agri,altopiano,nessun dissesto evidente,...,2024.0,"Trausi, Pier Pasquale",GLOSSA Srl,"maddaluno, corrado",responsabile coordinamento delle attività,GLOSSA Srl,2.0,Masseria Rubalo,"Casa di abitazione rurale di tipo ""collinare"" ...",0
3,7aea73d3-286c-4fa1-aade-0883bd9f9c78,182399,EDIFICIO CON ANNESSI,cascina,2,Friuli-Venezia Giulia,UD,Comune di Pavia di Udine,pianura,alluvione,...,2024.0,"Rulli, Eduardo",Atlante Group SPA,"Parisi, Valeria",responsabile coordinamento delle attività,Atlante Group s.r.l.,2.0,,"Oltre al corpo di fabbrica principale, a compl...",0
4,d8d256d7-e480-40c6-bd8a-0d8d4dd66f45,310548,COMPLESSO,casa a edifici affiancati,2,Abruzzo,TE,Castelli,collina,nessun dissesto evidente,...,2025.0,"Frezzini, Luca",Unimol,"Alessandria, Francesco",responsabile coordinamento delle attività,Cles,2.0,Colledoro,Edificio a pianta rettangolare orientato in di...,0


In [6]:
#count rows and where gaussian column is 1

print("number of rows: {}".format(len(df_new)), "number of data with gaussian splat: {}".format(len(df_new[df_new['gaussian'] == '1'])))


number of rows: 44363 number of data with gaussian splat: 403


In [7]:
df_new["stato_conservazione"].value_counts()

stato_conservazione
discreto                14869
buono                   11337
mediocre                10023
cattivo                  4504
pessimo                  2510
                         1100
dato non disponibile       20
Name: count, dtype: int64

In [8]:
#print the last 50 titles from the schede that have buono or discreto stato_conservazione AND a non empty titolo
df_good_condition = df_new[df_new["stato_conservazione"].isin(["buono", "discreto"]) & (df_new["titolo"].str.strip() != "")]
print(f"Number of schede with buono or discreto stato_conservazione and non empty titolo: {len(df_good_condition):,}")
print("\nLast 50 titles:")
for i, titolo in enumerate(df_good_condition["titolo"].tail(50), start=1):
    print(f"{titolo}") 

Number of schede with buono or discreto stato_conservazione and non empty titolo: 5,598

Last 50 titles:
La Concia
Masseria Montanaro
Case dell'Oliveto
Casale del Fornaccio
Casino Dattilo
Case Pozzello
Casale della Vannina
C.Le Pagliarini
Cascina Sorianino
ex Ost.a dell'Ellera
Masseria Magistro
Casa Agrillusa
Casa Contrada Pratelli
Casale della Volpe
Case Morolo
C. Facchinaccia
C.Le Testaccio
Cascina Cantalupo
Villa Achille Albanese
corte Preatoni
Casa Ottaviani
Casa Cepa
Casone Bianco
Cascina Ulivieri
Casa Ferrara
Villa Spina
Cascina Pedaggera
Casa Firriato-Imbornone-Napolitani
Casa Piacentino
Casa Rizzo-Muegen
Villa Trabia
Tenuta Borgia
Case Scuderi
Baglio Ingardia Fontansalsa
Palazzo Viani Tagliavacca
Villa Serraino
Baglio Fontanasalsa
Cascina Ghiringhella
Podere Casa Grande
Casa Ingardia Misiliscemi
Baglio in Via Federico II Stupor Mundi -Nubia
Il baglio di Pantelleria
Cascina Scanna
Baglio Sanacore
Villa Funtanazzi
Casa Messina
Villa Raffo
Casa Gugliatore
Casa S. Iorio
Villa Jacon

In [10]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

df = pd.read_csv("dati_schede_min.csv")

LANDSCAPE_MAP = {
    "costa (bassa)": "costa",
    "costa (alta, falesie)": "costa",
    "valle": "valle e fondovalle",
    "fondovalle": "valle e fondovalle",
    "conca intramontana": "valle e fondovalle",
    "pedemontano": "valle e fondovalle",
    "lagunare": "zone umide e acque interne",
    "lacustre": "zone umide e acque interne",
    "palustre": "zone umide e acque interne",
    "foce fluviale": "zone umide e acque interne",
    "montagna": "montagna",
    "versante ripido": "montagna",
    "crinale/dorsale": "montagna",
    "versante a debole pendenza": "collina",
    "collina": "collina",
    "carsico (doline, cavità ipogee)": "carsico",
    "altopiano": "altopiano",
    "pianura": "pianura",
}

raw_counts = df["contesto_paesaggistico"].value_counts()

df["contesto_paesaggistico_ridotto"] = df["contesto_paesaggistico"].map(LANDSCAPE_MAP)

reduced_counts = df["contesto_paesaggistico_ridotto"].value_counts()
print("\n=== contesto_paesaggistico_ridotto (x8) ===")
print(reduced_counts.to_string())




=== contesto_paesaggistico_ridotto (x8) ===
contesto_paesaggistico_ridotto
collina                       16961
pianura                       14236
montagna                       5412
valle e fondovalle             4972
altopiano                       917
costa                           400
zone umide e acque interne      230
carsico                         134


In [11]:
# give me a list of first 10 id_schede that have contesto paesaggistico = "altopiano" and gaussian as 1
df_new[(df_new["contesto_paesaggistico"] == "altopiano") & (df_new["gaussian"] == "1")]["id_scheda_originale"].head(10)

1395     dbb213a4-c091-4685-bb2d-27d192028206
6143     d2ab0fd0-84e7-402f-9a31-2a1c03155be0
8901     ac2e862f-7ccf-49ce-9c2b-a849c94d8674
19748    4b7752de-622e-4116-ab78-813ef608958b
24259    30a1ac31-b90c-4bb7-a875-ab3b12f82f09
28385    309ce237-14f3-4843-9935-3d6bf014b530
42468    15878fc6-895c-46e1-9c44-eebdf82ac276
42538    036eb456-54ba-4155-9fec-99a26d9a8aaf
42989    bd5cc795-5465-43a3-b491-04adeb030a6d
43847    d6b9a4ee-0565-4c63-8a98-c5c18549081e
Name: id_scheda_originale, dtype: str

In [ ]:
# get 4 random rows with contensto apesaggistico altop